In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import windows
from ipywidgets import RadioButtons, VBox, HBox, interactive_output
from IPython.display import display

def analyze_window(window_type):
    plt.close('all')
    
    # ----------------------------------------------------
    # ASSUMPTIONS & CONSTANTS (According to Harris' landmark paper)
    # ----------------------------------------------------
    N = 128
    alpha_val = 5.0
    n_fft = 16384

    # 1. WINDOW GENERATION
    if window_type == 'Rectangular':
        w = windows.boxcar(N)
    elif window_type == 'Bartlett':
        w = windows.bartlett(N)
    elif window_type == 'Hann':
        w = windows.hann(N, sym=True)
    elif window_type == 'Hamming':
        w = windows.hamming(N, sym=True)
    elif window_type == 'Blackman':
        w = windows.blackman(N, sym=True)
    elif window_type == 'Kaiser':
        w = windows.kaiser(N, beta=alpha_val)
    else:
        w = windows.boxcar(N)

    # 2. THEORETICAL METRICS COMPUTATION
    sum_w = np.sum(w)
    sum_w2 = np.sum(w**2)
    g_coh = sum_w / N
    
    b_eq = N * (sum_w2 / (sum_w**2))
    
    g_p = 1.0 / b_eq
    l_p_db = 10.0 * np.log10(b_eq * 2.0)

    # 3. SPECTRUM & SIDELOBE COMPUTATION
    w_padded = np.zeros(n_fft)
    w_padded[:N] = w
    W_spec = np.fft.fftshift(np.fft.fft(w_padded, n_fft))
    
    freqs_bins = np.fft.fftshift(np.fft.fftfreq(n_fft, d=1/N))
    W_mag_dB = 20.0 * np.log10(np.abs(W_spec) / np.abs(W_spec).max() + 1e-12)

    center_idx = n_fft // 2
    main_lobe_width_samples = int(np.ceil(2 * b_eq * (n_fft / N)))
    
    left_side = W_mag_dB[:center_idx - main_lobe_width_samples]
    right_side = W_mag_dB[center_idx + main_lobe_width_samples:]
    max_sidelobe_dB = max(left_side.max(), right_side.max()) if (len(left_side) > 0 and len(right_side) > 0) else 0

    # 4. PLOTTING
    fig = plt.figure(figsize=(14, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.2, 1], wspace=0.3)

    # Left plot: Spectrum in dB
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(freqs_bins, W_mag_dB, 'b-', lw=1.5)
    
    title_str = f"Spectrum of {window_type} (N={N})"
    if window_type == 'Kaiser':
        title_str += f", α={alpha_val}"
    
    ax1.set_title(title_str, fontsize=11, fontweight='bold')
    ax1.set_xlabel("Frequency [Bins]", fontsize=9)
    ax1.set_ylabel("Magnitude [dB]", fontsize=9)
    ax1.set_ylim(-100, 5)
    ax1.set_xlim(-8, 8)
    ax1.grid(True, linestyle='--', alpha=0.6)

    # Right plot: Bar chart of Harris metrics
    ax2 = fig.add_subplot(gs[0, 1])
    metrics_names = ['B_eq (bins)', 'G_coh', 'H_side (-dB scaled)']
    metrics_values = [b_eq, g_coh, abs(max_sidelobe_dB) / 10]

    ax2.bar(metrics_names, metrics_values, color=['#1f77b4', '#2ca02c', '#d62728'], alpha=0.85)
    ax2.set_title("Performance Coefficients (Harris Table 1)", fontsize=11, fontweight='bold')
    ax2.set_ylabel("Normalized Value / Scale", fontsize=9)
    ax2.grid(True, axis='y', linestyle='--', alpha=0.6)

    y_max = max(metrics_values)
    ax2.set_ylim(0, y_max * 1.15)

    ax2.text(0, b_eq + 0.05, f"{b_eq:.2f}", ha='center', fontsize=9, fontweight='bold')
    ax2.text(1, g_coh + 0.05, f"{g_coh:.2f}", ha='center', fontsize=9, fontweight='bold')
    ax2.text(2, (abs(max_sidelobe_dB) / 10) + 0.05, f"{max_sidelobe_dB:.1f} dB", ha='center', fontsize=9, fontweight='bold')

    plt.show()

    # 5. TEXT OUTPUT SUMMARY
    print("=" * 75)
    print(f" WINDOW PERFORMANCE METRICS: {window_type.upper()} (N = {N})" + (f", α = {alpha_val}" if window_type == 'Kaiser' else ""))
    print("=" * 75)
    print(f"• Coherent Gain (G_coh)             : {g_coh:.4f}")
    print(f"• Equivalent Noise Bandwidth (B_eq) : {b_eq:.4f} bins")
    print(f"• Processing Loss (L_p)             : {l_p_db:.4f} dB")
    print(f"• 1st Sidelobe Level (H_side)       : {max_sidelobe_dB:.2f} dB")
    print("-" * 75)
    print("Note: Minor discrepancies compared to theoretical tables typically arise from")
    print("discrete sampling effects, choice of window symmetry, and FFT bin resolution limits.")
    print("=" * 75)

# Widget Setup
window_radio = RadioButtons(
    options=['Rectangular', 'Bartlett', 'Hann', 'Hamming', 'Blackman', 'Kaiser'],
    value='Rectangular',
    description='Window:',
    disabled=False
)

controls = VBox([window_radio])
out = interactive_output(analyze_window, {'window_type': window_radio})

display(HBox([controls, out]))